# Лабораторная №4 — CIFAR-10 и ResNet

Полный учебный цикл по методичке: 1. загрузка → 2. визуализация → 3. предобработка → 4. архитектура ResNet и заполнение пропусков → 5. обучение → 6. тестирование → 7. визуализация. CIFAR-10 скачивается автоматически. Для Colab рекомендуется GPU.

## 1. Загрузка данных CIFAR-10

In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
tf.keras.utils.set_random_seed(42)
(x_train,y_train),(x_test,y_test)=cifar10.load_data()
print(x_train.shape,x_test.shape)

## 2. Вывод загруженных изображений

In [ ]:
names=["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
fig,axs=plt.subplots(4,4,figsize=(9,9))
for i,ax in enumerate(axs.flat):
    ax.imshow(x_train[i]);ax.set_title(names[int(y_train[i,0])]);ax.axis("off")
plt.tight_layout();plt.show()

## 3. Предобработка — нормализация и one-hot кодирование

In [ ]:
x_train=x_train.astype("float32")/255.
x_test=x_test.astype("float32")/255.
y_train_oh=keras.utils.to_categorical(y_train,10)
y_test_oh=keras.utils.to_categorical(y_test,10)
print(x_train.min(),x_train.max(),y_train_oh.shape)

## 4. ResNet — заполняем все пропуски методички

Реализуем residual block с shortcut, свёртками Conv2D, BatchNormalization, ReLU и Add. При изменении числа каналов или размера используем 1×1 projection. Составляем сеть из residual-блоков и классификатора Softmax.

In [ ]:
def residual_block(x,filters,stride=1):
    shortcut=x
    z=layers.Conv2D(filters,3,strides=stride,padding="same",use_bias=False)(x)
    z=layers.BatchNormalization()(z)
    z=layers.Activation("relu")(z)
    z=layers.Conv2D(filters,3,padding="same",use_bias=False)(z)
    z=layers.BatchNormalization()(z)
    if stride!=1 or int(shortcut.shape[-1])!=filters:
        shortcut=layers.Conv2D(filters,1,strides=stride,padding="same",use_bias=False)(shortcut)
        shortcut=layers.BatchNormalization()(shortcut)
    z=layers.Add()([z,shortcut])
    return layers.Activation("relu")(z)
inputs=keras.Input((32,32,3))
x=layers.Conv2D(32,3,padding="same",use_bias=False)(inputs)
x=layers.BatchNormalization()(x)
x=layers.Activation("relu")(x)
x=residual_block(x,32)
x=residual_block(x,32)
x=residual_block(x,64,stride=2)
x=residual_block(x,64)
x=residual_block(x,128,stride=2)
x=residual_block(x,128)
x=layers.GlobalAveragePooling2D()(x)
outputs=layers.Dense(10,activation="softmax")(x)
model=keras.Model(inputs,outputs,name="ResNet_CIFAR10")
model.summary()

## 4.1. Подготовка к обучению — compile

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="categorical_crossentropy",metrics=["accuracy"])

## 5. Обучение ResNet

10 эпох — быстрая проверка. Для более высокого качества увеличьте `EPOCHS` до 30–50. Данные валидации отделяются только от обучающей выборки.

In [ ]:
EPOCHS=10
history=model.fit(x_train,y_train_oh,batch_size=128,epochs=EPOCHS,validation_split=0.1,verbose=1)

## 6. Тестирование

In [ ]:
loss,acc=model.evaluate(x_test,y_test_oh,verbose=0)
print(f"Test loss={loss:.4f}, accuracy={acc:.4f}")

## 7. Результат: графики и предсказания

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,k in zip(axs,["loss","accuracy"]):
    ax.plot(history.history[k],label="train")
    ax.plot(history.history["val_"+k],label="validation")
    ax.set_title(k);ax.legend()
plt.show()
prob=model.predict(x_test[:16],verbose=0)
fig,axs=plt.subplots(4,4,figsize=(10,10))
for i,ax in enumerate(axs.flat):
    p=int(prob[i].argmax());t=int(y_test[i,0])
    ax.imshow(x_test[i]);ax.set_title(f"{names[p]} / {names[t]}",color="green" if p==t else "red");ax.axis("off")
plt.tight_layout();plt.show()

## 7.1. Матрица ошибок и выводы

In [ ]:
pred=model.predict(x_test,verbose=0).argmax(axis=1)
true=y_test.ravel()
print(classification_report(true,pred,target_names=names))
ConfusionMatrixDisplay.from_predictions(true,pred,display_labels=names,xticks_rotation=45)
plt.show()

**Вывод:** после выполнения запишите фактическую точность, наиболее часто путаемые классы и сравните train/validation. Метрики не подставляются заранее.